# HR-VITON Inference Server — Google Colab
**Dollaby Graduation Project**

Runs HR-VITON on Colab's T4 GPU and exposes it via an **ngrok HTTPS tunnel**.
Your backend calls it using `VITON_MODE=colab`.

### Workflow
1. Run this notebook (all cells top to bottom)
2. Copy the ngrok URL printed at the end
3. Add to `backend/.env`:
   ```
   VITON_MODE=colab
   COLAB_VITON_URL=https://xxxx.ngrok-free.app
   ```
4. Restart your FastAPI backend
5. Keep this notebook alive during your demo

> **Enable GPU first:** Runtime → Change runtime type → T4 GPU

## Step 1 — Verify GPU

In [1]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

print(f"✓ GPU : {torch.cuda.get_device_name(0)}")
print(f"✓ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

RuntimeError: No GPU detected. Go to Runtime → Change runtime type → T4 GPU

## Step 2 — Install Dependencies

In [ ]:
import subprocess, sys

packages = [
    'fastapi', 'uvicorn[standard]', 'pyngrok', 'nest-asyncio',
    'rembg',               # cloth background removal
    'transformers',        # SegFormer human parsing
    'gdown', 'einops',
    # mediapipe removed: solutions API dropped in versions with Python 3.12 support
    # body mask is derived from SegFormer parse instead (same quality, no dependency)
]

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + packages, check=True)

# Force-reinstall Pillow to fix version fragmentation from rembg install
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', 'Pillow'], check=True)

print('✓ All dependencies installed')

## Step 3 — Mount Drive & Load Checkpoints

In [ ]:
import os, shutil, gdown

LOCAL_CKPT = '/kaggle/working/checkpoints'
os.makedirs(LOCAL_CKPT, exist_ok=True)

# ── TOCG ──────────────────────────────────────────────────────────────────
TOCG_INPUT = '/kaggle/input/datasets/kareemtahaaa/gp-fine-tune/tocg_finetuned.pth'
tocg_local  = f'{LOCAL_CKPT}/tocg.pth'

if os.path.exists(TOCG_INPUT):
    shutil.copy(TOCG_INPUT, tocg_local)
    print('✓ TOCG — using fine-tuned checkpoint from Kaggle dataset')
else:
    print(f'⚠️  TOCG not found at {TOCG_INPUT}')
    print('   Falling back to official pretrained...')
    gdown.download(id='1XJTCdRBOPVgVTmqzhVGFAgMm2NLkw5uQ', output=tocg_local, quiet=False)
    print('✓ TOCG (pretrained) downloaded')

# ── Generator ─────────────────────────────────────────────────────────────
gen_local = f'{LOCAL_CKPT}/gen.pth'
if not os.path.exists(gen_local):
    print('Downloading Image Generator (official pretrained)...')
    gdown.download(id='1T5_YDUhYSSKPC_nZMk2NeC-XXUFoYeNy', output=gen_local, quiet=False)
print('✓ Generator checkpoint ready')

## Step 4 — Clone HR-VITON & Load Models

In [ ]:
import subprocess, sys

HRVITON_DIR = '/kaggle/working/HR-VITON'

if not os.path.exists(HRVITON_DIR):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/sangyun884/HR-VITON.git', HRVITON_DIR], check=True)
    print('✓ HR-VITON cloned')
else:
    print('✓ HR-VITON already present')

sys.path.insert(0, HRVITON_DIR)

# Install HR-VITON requirements
req = f'{HRVITON_DIR}/requirements.txt'
if os.path.exists(req):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', req])

In [ ]:
import torch
import argparse
import numpy as np
from PIL import Image

from networks import ConditionGenerator, load_checkpoint
from network_generator import SPADEGenerator

device = 'cuda'
IMG_H, IMG_W = 256, 192

opt = argparse.Namespace(
    fine_width=IMG_W, fine_height=IMG_H,
    semantic_nc=13, output_nc=13, ngf=96,
    num_D=2, spectral=True,
    norm_G='spectralaliasinstance',
    Ddownx2=True, Ddropout=True,
    occlusion=True, warp_feature='T1',
    out_layer='relu',
    gen_semantic_nc=7,
    gpu_ids=[0],
    norm_type='instance', upsample='bilinear',
    num_upsampling_layers='most',
    no_ganFeat_loss=False, no_vgg_loss=False,
    use_vae=False, contain_dontcare_label=False,
    crop_size=IMG_H,
    ndf=64, norm_D='spectralinstance', n_layers_D=3,
    cuda=True,
)

print('Loading TOCG...')
tocg = ConditionGenerator(opt, input1_nc=4, input2_nc=16, output_nc=13, ngf=opt.ngf)
load_checkpoint(tocg, f'{LOCAL_CKPT}/tocg.pth', opt)
tocg.to(device).eval()
n_tocg = sum(p.numel() for p in tocg.parameters())
print(f'✓ TOCG loaded  ({n_tocg/1e6:.1f}M params)')

# Official gen checkpoint was trained with ngf=64, not 96
opt.ngf = 64
print('Loading Generator...')
gen = SPADEGenerator(opt, 3 + 3 + 3)
load_checkpoint(gen, f'{LOCAL_CKPT}/gen.pth', opt)
gen.to(device).eval()
n_gen = sum(p.numel() for p in gen.parameters())
print(f'✓ Generator loaded  ({n_gen/1e6:.1f}M params)')

## Step 5 — Load Preprocessing Models

In [ ]:
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor

print('Loading human parser (SegFormer)...')
parse_processor = SegformerImageProcessor.from_pretrained('mattmdjaga/segformer_b2_clothes')
parse_model     = SegformerForSemanticSegmentation.from_pretrained('mattmdjaga/segformer_b2_clothes')
parse_model.to(device).eval()
print('✓ Human parser loaded')
print('✓ Pose/DensePose: derived from SegFormer parse (no MediaPipe needed)')

## Step 6 — Preprocessing Pipeline

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image
from torchvision import transforms

SEG_REMAP = {
    0: 0,   # background
    2: 1,   # hair
    11: 2,  # face
    15: 3,  # right-arm
    14: 4,  # left-arm
    4: 5,   # upper-clothes
    7: 5,   # dress → upper-clothes
    12: 6,  # left-leg
    13: 7,  # right-leg
    9: 8,   # left-shoe
    10: 9,  # right-shoe
    6: 11,  # pants
    5: 11,  # skirt → pants
    1: 12,  # hat → others
    3: 12,  # glasses → others
    8: 12,  # belt → others
    16: 12, # bag → others
    17: 12, # scarf → others
}


def pil_to_tensor(img, normalize=True):
    arr = np.array(img).astype(np.float32)
    if arr.ndim == 2:
        arr = arr[:, :, np.newaxis]
    t = torch.from_numpy(arr.transpose(2, 0, 1))
    return ((t / 127.5 - 1.0) if normalize else t / 255.0).unsqueeze(0).to(device)


def preprocess_cloth(cloth_pil: Image.Image):
    from rembg import remove
    cloth_rgba = remove(cloth_pil.convert('RGB'))
    arr        = np.array(cloth_rgba)
    cloth_rgb  = Image.fromarray(arr[:, :, :3]).resize((IMG_W, IMG_H), Image.LANCZOS)
    mask_np    = (arr[:, :, 3] > 10).astype(np.uint8) * 255
    cloth_mask = Image.fromarray(mask_np).resize((IMG_W, IMG_H), Image.NEAREST)
    return cloth_rgb, cloth_mask


def parse_human(person_pil: Image.Image):
    resized = person_pil.resize((IMG_W, IMG_H))
    inputs  = parse_processor(images=resized, return_tensors='pt').to(device)
    with torch.no_grad():
        logits = parse_model(**inputs).logits
    pred = F.interpolate(logits, size=(IMG_H, IMG_W), mode='bilinear', align_corners=False)
    parse_raw = pred.argmax(dim=1).squeeze().cpu().numpy().astype(np.uint8)
    parse_13  = np.zeros_like(parse_raw)
    for src, dst in SEG_REMAP.items():
        parse_13[parse_raw == src] = dst
    return parse_13


def make_agnostic(person_pil: Image.Image, parse_np: np.ndarray):
    person_np  = np.array(person_pil.resize((IMG_W, IMG_H)))
    agnostic   = person_np.copy()
    upper_mask = (parse_np == 5)
    if upper_mask.any():
        agnostic[upper_mask] = [128, 128, 128]
    return Image.fromarray(agnostic)


def make_densepose_approx(parse_np: np.ndarray):
    """
    Approximate DensePose UV map derived from SegFormer body segmentation.
    No MediaPipe needed — uses parse_np (already computed) as the body mask.
    """
    body_mask = (parse_np > 0)           # everything except background (class 0)
    yy, xx    = np.mgrid[0:IMG_H, 0:IMG_W]
    dp        = np.zeros((IMG_H, IMG_W, 3), dtype=np.float32)
    dp[:, :, 0] = (xx / IMG_W) * body_mask   # U
    dp[:, :, 1] = (yy / IMG_H) * body_mask   # V
    dp[:, :, 2] = body_mask.astype(float)     # presence
    return dp


print('✓ Preprocessing functions ready')

## Step 7 — Inference Function

In [ ]:
import io, base64

@torch.no_grad()
def run_hrviton(person_pil: Image.Image, garment_pil: Image.Image) -> Image.Image:
    person_pil  = person_pil.convert('RGB').resize((IMG_W, IMG_H), Image.LANCZOS)
    garment_pil = garment_pil.convert('RGB').resize((IMG_W, IMG_H), Image.LANCZOS)

    cloth_rgb, cloth_mask = preprocess_cloth(garment_pil)
    parse_np              = parse_human(person_pil)
    agnostic_pil          = make_agnostic(person_pil, parse_np)
    dense_approx          = make_densepose_approx(parse_np)   # uses SegFormer body mask

    # input1: cloth (3ch) + cloth_mask (1ch) = 4ch
    cloth_t = pil_to_tensor(cloth_rgb)
    mask_np = np.array(cloth_mask)[:, :, np.newaxis].astype(np.float32) / 255.0
    mask_t  = torch.from_numpy(mask_np.transpose(2, 0, 1)).unsqueeze(0).to(device)
    input1  = torch.cat([cloth_t, mask_t], dim=1)

    # input2: parse one-hot (13ch) + densepose (3ch) = 16ch
    parse_oh = torch.zeros(1, 13, IMG_H, IMG_W, device=device)
    for c in range(13):
        parse_oh[0, c] = torch.from_numpy((parse_np == c).astype(np.float32)).to(device)
    dense_t = torch.from_numpy(dense_approx.transpose(2, 0, 1)).unsqueeze(0).to(device)
    input2  = torch.cat([parse_oh, dense_t], dim=1)

    _, _, warped_cloth, _ = tocg(opt, input1, input2)

    agnostic_t = pil_to_tensor(agnostic_pil)
    gen_input  = torch.cat([agnostic_t, dense_t, warped_cloth], dim=1)

    try:
        result_t = gen(gen_input)
        if isinstance(result_t, (list, tuple)):
            result_t = result_t[0]
    except Exception:
        blend_mask = mask_t.expand_as(warped_cloth)
        result_t   = warped_cloth * blend_mask + agnostic_t * (1 - blend_mask)

    out    = ((result_t.squeeze(0).cpu() + 1.0) / 2.0).clamp(0.0, 1.0)
    out_np = (out.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    return Image.fromarray(out_np)


print('✓ Inference function ready')

# Quick smoke test
dummy_p = Image.fromarray(np.random.randint(100, 200, (512, 384, 3), dtype=np.uint8))
dummy_g = Image.fromarray(np.random.randint(50,  180, (256, 256, 3), dtype=np.uint8))
result  = run_hrviton(dummy_p, dummy_g)
print(f'✓ Smoke test passed — output size: {result.size}')

## Step 8 — FastAPI Server

In [ ]:
import threading, io, base64
import uvicorn, nest_asyncio
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import JSONResponse
from PIL import Image

nest_asyncio.apply()
app = FastAPI(title='HR-VITON Server')


@app.get('/health')
def health():
    return {
        'status': 'ok',
        'model': 'HR-VITON (fine-tuned TOCG)',
        'gpu': torch.cuda.get_device_name(0),
        'vram_gb': round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
    }


@app.post('/tryon')
async def tryon(
    person_image:  UploadFile = File(...),
    garment_image: UploadFile = File(...),
    category:      str = Form(default='upperbody'),
    num_steps:     str = Form(default='20'),
):
    try:
        person_pil  = Image.open(io.BytesIO(await person_image.read())).convert('RGB')
        garment_pil = Image.open(io.BytesIO(await garment_image.read())).convert('RGB')

        result_pil = run_hrviton(person_pil, garment_pil)

        buf = io.BytesIO()
        result_pil.save(buf, format='JPEG', quality=92)
        img_b64 = base64.b64encode(buf.getvalue()).decode()

        return JSONResponse({'success': True, 'image_base64': img_b64})

    except Exception as exc:
        import traceback
        tb = traceback.format_exc()
        return JSONResponse({'success': False, 'error': str(exc), 'traceback': tb}, status_code=500)


SERVER_PORT = 8080

def _run():
    uvicorn.run(app, host='0.0.0.0', port=SERVER_PORT, log_level='warning')

t = threading.Thread(target=_run, daemon=True)
t.start()

import time; time.sleep(2)  # wait for startup
print(f'✓ FastAPI server listening on port {SERVER_PORT}')

## Step 9 — Start ngrok Tunnel

1. Get a **free ngrok account** at https://ngrok.com
2. Copy your auth token from https://dashboard.ngrok.com/get-started/your-authtoken
3. Paste it in `NGROK_TOKEN` below

In [ ]:
from pyngrok import ngrok

# ← Paste your ngrok auth token here (free at ngrok.com)
NGROK_TOKEN = ''

if not NGROK_TOKEN:
    print('⚠️  Paste your ngrok token in NGROK_TOKEN above.')
    print('   Get one free at: https://dashboard.ngrok.com/get-started/your-authtoken')
    raise SystemExit()

ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()   # kill any previous tunnels

tunnel = ngrok.connect(SERVER_PORT, 'http')
public_url = tunnel.public_url

print('\n' + '='*60)
print('  ✓  HR-VITON server is LIVE!')
print('='*60)
print(f'  URL : {public_url}')
print('='*60)
print()
print('Add these two lines to backend/.env:')
print()
print('  VITON_MODE=colab')
print(f'  COLAB_VITON_URL={public_url}')
print()
print('Then restart your FastAPI backend.')
print('Keep this notebook running during your demo.')
print('(URL changes each session — update .env when you restart)')

## Step 10 — Keep Alive

Colab disconnects after ~90 minutes of inactivity. Run this cell to stay alive while demoing.

In [ ]:
import time, requests, datetime

print('Keep-alive loop started. Stop with Runtime → Interrupt execution.')
print()

while True:
    try:
        r = requests.get(f'http://localhost:{SERVER_PORT}/health', timeout=5)
        data = r.json()
        print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] server OK — "
              f"GPU: {data.get('gpu','?')}  VRAM: {data.get('vram_gb','?')} GB")
    except Exception as e:
        print(f'Server check failed: {e}')
    time.sleep(60)